# Phase 10: Fold Robustness Validation

## 1. Packages and paths

In [7]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from itertools import product
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

warnings.filterwarnings('default')
np.random.seed(36)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths
PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models_folds'
PHASE10_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase10_fold_robustness'
PHASE10_OUTPUT.mkdir(parents=True, exist_ok=True)

#Constants
DATASET='2007'
FOLDS=['Fold1', 'Fold2', 'Fold3']
MODELS=['pointwise', 'pairwise', 'lightgbm']
PIPELINES=['raw', 'global', 'per_query']
EXPECTED_CONFIGS=len(MODELS) * len(PIPELINES)  # 9

BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'
MIN_SAMPLE_WARNING=10

warnings_issued=[]

print('='*80)
print('PHASE 10: FOLD ROBUSTNESS VALIDATION')
print('='*80)
print(f'Output: {PHASE10_OUTPUT}')
print(f'Dataset: MQ{DATASET}')
print(f'Folds: {FOLDS}')
print(f'Expected configs/fold: {EXPECTED_CONFIGS}')
print(f'Baseline: {BASELINE_MODEL}_{BASELINE_PIPELINE}')
print('='*80)

PHASE 10: FOLD ROBUSTNESS VALIDATION
Output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase10_fold_robustness
Dataset: MQ2007
Folds: ['Fold1', 'Fold2', 'Fold3']
Expected configs/fold: 9
Baseline: pointwise_raw


## 2. Util functions

In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    make_fail_flag,
    check_sample_size,
    safe_k,
    compute_failure_at_k,
    mcnemar_test,
    bh_fdr,
)

In [9]:
def compute_score_gap(pred_df: pd.DataFrame, qids: set, k: int=5, relevance_threshold: int=1) -> dict:
    """
    Score gap=(best relevant score)-(score at rank k).
    Returns {qid: gap or np.nan}.
    """
    gaps={}
    for qid in qids:
        q_docs=pred_df[pred_df["qid"] == qid].copy()
        if len(q_docs)==0:
            gaps[qid]=np.nan
            continue

        q_docs=q_docs.sort_values("score", ascending=False)

        relevant=q_docs[q_docs["label"]>=relevance_threshold]
        if len(relevant)==0:
            gaps[qid]=np.nan
            continue

        best_rel=float(relevant["score"].max())

        k_actual=min(k, len(q_docs))
        if k_actual<=0:
            gaps[qid]=np.nan
            continue

        score_at_k=float(q_docs.iloc[k_actual-1]["score"])
        gaps[qid]=best_rel-score_at_k

    return gaps

print("imported src.utils + defined compute_score_gap")

imported src.utils + defined compute_score_gap


## 3. Loading Phase 6 Artifacts

In [10]:
print('\n'+'='*80)
print('LOADING PHASE 6 ARTIFACTS')
print('='*80)

all_fold_artifacts={}

for fold in FOLDS:
    print(f'\n{fold}:')
    artifacts={'query_metrics': {}, 'predictions': {}}
    
    for model, pipeline in product(MODELS, PIPELINES):
        key=f'{model}_{pipeline}_{DATASET}'
        qm_file=PHASE6_OUTPUT / fold/f'{key}_query_metrics.csv'
        pred_file=PHASE6_OUTPUT / fold/f'{key}_predictions.csv'
        
        if not qm_file.exists() or not pred_file.exists():
            print(f'Missing: {key}')
            continue
        
        qm=pd.read_csv(qm_file)
        pred=pd.read_csv(pred_file)
        
        #Proper validation with flag
        ok=True
        for col in ['qid', 'num_docs', 'num_relevant_1', 'Failure@5_primary']:
            if col not in qm.columns:
                print(f'{key} missing column: {col}')
                ok=False
        for col in ['qid', 'label', 'score']:
            if col not in pred.columns:
                print(f'{key} missing column: {col}')
                ok=False
        if not ok:
            continue
        
        #Coercing dtypes
        qm['qid']=qm['qid'].astype(int)
        pred['qid']=pred['qid'].astype(int)
        pred['label']=pred['label'].astype(int)
        pred['score']=pred['score'].astype(float)
        
        artifacts['query_metrics'][key]=qm
        artifacts['predictions'][key]=pred
    
    if len(artifacts['query_metrics'])>0:
        all_fold_artifacts[fold]=artifacts
        print(f'{len(artifacts["query_metrics"])} configs')
    else:
        msg=f'{fold}: no configs loaded'
        warnings_issued.append(msg)
        print(f'{msg}')

print(f'\nLoaded {len(all_fold_artifacts)}/{len(FOLDS)} folds')
print('='*80)


LOADING PHASE 6 ARTIFACTS

Fold1:
9 configs

Fold2:
9 configs

Fold3:
9 configs

Loaded 3/3 folds


## 4. Step 1: Baseline Failure Rates

In [11]:
print('\n'+'='*80)
print('STEP 1: BASELINE FAILURE RATES')
print('='*80)

baseline_rates=[]

for fold in FOLDS:
    if fold not in all_fold_artifacts:
        print(f'{fold}: skipped (not loaded)')
        continue
    
    baseline_key=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}_{DATASET}'
    if baseline_key not in all_fold_artifacts[fold]['query_metrics']:
        msg=f'{fold}: missing {baseline_key}'
        warnings_issued.append(msg)
        print(f'{msg}')
        continue
    
    qm=all_fold_artifacts[fold]['query_metrics'][baseline_key]
    evaluable=qm[qm['num_relevant_1']>0]
    n_evaluable=len(evaluable)
    
    if n_evaluable==0:
        print(f'{fold}: no evaluable queries')
        continue
    
    fail_flag=make_fail_flag(evaluable['Failure@5_primary'])
    n_failures=int((fail_flag==1).sum())
    pct_failures=100 * n_failures / n_evaluable
    
    baseline_rates.append({
        'fold': fold,
        'n_evaluable': n_evaluable,
        'n_failures': n_failures,
        'pct_failures': float(pct_failures)
    })
    print(f'{fold}: {n_failures}/{n_evaluable} ({pct_failures:.1f}%)')

baseline_df=pd.DataFrame(baseline_rates).sort_values('fold')
if len(baseline_df)>0:
    display(baseline_df)
    baseline_df.to_csv(PHASE10_OUTPUT / 'phase10_fold_failure_rates.csv', index=False)
    print('\nSaved: phase10_fold_failure_rates.csv')
else:
    baseline_df = pd.DataFrame()
    print('No baseline rates computed')


STEP 1: BASELINE FAILURE RATES
Fold1: 44/290 (15.2%)
Fold2: 39/283 (13.8%)
Fold3: 59/295 (20.0%)


,fold,n_evaluable,n_failures,pct_failures
0,Fold1,290,44,15.1724
1,Fold2,283,39,13.7809
2,Fold3,295,59,20.0000



Saved: phase10_fold_failure_rates.csv


Here we are looking at how the baseline model performs on each fold separately. We are only counting *evaluable* queries, meaning queries where there was at least one relevant document somewhere in the list. So these are the cases where the model actually had a fair chance to succeed.

What this tells us is that the baseline model fails on roughly **14–20%** of evaluable queries depending on the fold. The numbers are not identical, which is expected because each fold contains slightly different training and test splits. 

Fold3 seems a bit harder since it has a higher failure rate, while Fold2 looks slightly easier. But overall, the failure rates are in a similar range. There is some variation, but nothing extreme or suspicious.

This gives us a stable starting point. Now we know how often the baseline fails in each fold, and we can later check whether persistent failures are consistent across folds or just due to random split differences.